<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/3D_conf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 45.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

def process_and_embed_mols(dataframe, smiles_col, id_col, output_writer, num_confs=5):
    """
    Scientifically accurate function to generate 3D ensembles.
    Isolates conformers into unique single-conformer molecule objects
    to guarantee flawless property assignment and zero index clashing.
    """
    unique_molecules_saved = 0

    for index, row in dataframe.iterrows():
        smiles = row[smiles_col]

        # Extract ID
        if id_col in dataframe.columns and pd.notna(row[id_col]):
            mol_id = str(row[id_col])
        else:
            mol_id = f"Mol_{index}"

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: Cannot parse SMILES for {mol_id}. Skipping.")
            continue

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # Enforce ETKDGv3 and chirality
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.enforceChirality = True

        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, params=params)
        if not conf_ids:
            print(f"Warning: 3D embedding failed for {mol_id}. Skipping.")
            continue

        molecule_had_successful_conformer = False

        for conf_id in conf_ids:
            # Maximize iterations (500) to prevent throwing away bulky Enamine chemical space
            opt_status = AllChem.MMFFOptimizeMolecule(mol, confId=conf_id, maxIters=500)

            if opt_status == 0:
                # 1. Create a clean, blank copy of the topology (no conformers attached)
                single_conf_mol = Chem.Mol(mol)
                single_conf_mol.RemoveAllConformers()

                # 2. Extract the specific optimized conformer from the parent molecule
                conf = mol.GetConformer(conf_id)

                # 3. Force internal ID to 0 so SDWriter reads it cleanly
                conf.SetId(0)

                # 4. Add the re-indexed conformer back into the isolated wrapper
                single_conf_mol.AddConformer(conf, assignId=False)

                # 5. Set distinct, isolated metadata properties that won't leak or overwrite
                single_conf_mol.SetProp("_Name", f"{mol_id}_conf_{conf_id}")
                single_conf_mol.SetProp("Parent_ID", mol_id)

                # 6. Safely write (it sees exactly one conformer sitting at index 0)
                output_writer.write(single_conf_mol)
                molecule_had_successful_conformer = True
            else:
                pass  # Suppress failed optimizations for clean terminal output

        if molecule_had_successful_conformer:
            unique_molecules_saved += 1

    return unique_molecules_saved


# Data Curation & Deduplication Pipeline


print("Loading datasets...")
sheet1_path = "/content/AA_Derivatives_Actives.csv"
sheet2_path = "/content/True_Negatives.csv"
sheet3_path = "/content/Enamine_VS_Hits_TrueNegs.csv"

df_sheet1_actives = pd.read_csv(sheet1_path)
df_neg = pd.read_csv(sheet2_path)
df_vs = pd.read_csv(sheet3_path)

# 1. Define subsets
df_sheet3_actives = df_vs[df_vs['Activity Tier'] != 'No Inhibition']
df_vs_inactives = df_vs[df_vs['Activity Tier'] == 'No Inhibition']

# 2. Extract raw SMILES sets for cross-referencing
sheet1_smiles = set(df_sheet1_actives['SMILES'])
sheet3_active_smiles = set(df_sheet3_actives['SMILES'])
all_active_smiles = sheet1_smiles | sheet3_active_smiles

# 3. Ensure Zero Leakage
# Drop any Sheet 3 Actives that are already in Sheet 1
df_sheet3_actives = df_sheet3_actives[~df_sheet3_actives['SMILES'].isin(sheet1_smiles)]

# Drop any Training Negatives that exist in ANY Active set
df_neg = df_neg[~df_neg['SMILES'].isin(all_active_smiles)]
df_vs_inactives = df_vs_inactives[~df_vs_inactives['SMILES'].isin(all_active_smiles)]

print("Data deduplication complete. Proceeding to 3D embedding...")


# 3D Embedding and Export


# Set 1: Training Actives (Sheet 1 + Sheet 3)
writer_train_actives = Chem.SDWriter("Training_Actives_3D.sdf")
act_count_1 = process_and_embed_mols(df_sheet1_actives, 'SMILES', 'Compound ID', writer_train_actives, num_confs=5)
act_count_2 = process_and_embed_mols(df_sheet3_actives, 'SMILES', 'Catalog ID', writer_train_actives, num_confs=5)
writer_train_actives.close()

total_actives = act_count_1 + act_count_2
print(f"Set 1 Complete: {total_actives} unique molecules saved for Training (ACTIVES).")

# Set 2: Training Negatives
writer_train_negatives = Chem.SDWriter("Training_Negatives_3D.sdf")
neg_count_s2 = process_and_embed_mols(df_neg, 'SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
neg_count_s3 = process_and_embed_mols(df_vs_inactives, 'SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
writer_train_negatives.close()

total_negatives = neg_count_s2 + neg_count_s3
print(f"Set 2 Complete: {total_negatives} unique molecules saved for Training (NEGATIVES).")



Loading datasets...
Data deduplication complete. Proceeding to 3D embedding...
Set 1 Complete: 85 unique molecules saved for Training (ACTIVES).
Set 2 Complete: 54 unique molecules saved for Training (NEGATIVES).
